In [ ]:
# run this notebook after 11_trim_ibd2_python
# use this notebook to merge trio DNM calls after standard QC + GIAB
# after this, run 13_determine_ibd2_trim_R

In [ ]:
library(dplyr)
library(bedtoolsr)
library(data.table)
library(ggplot2)

In [ ]:
# go through each trio pair  
trios$difs_file_exists = FALSE
trios$num_snps_raw = NA

In [ ]:
format_difs = function(d){
    d$chr = NA
    d$pos = NA
    chr_pos = gsub(x = d$locus, pattern = "chr", replace = "")
    d$chr = gsub(x = chr_pos, pattern = ":.*", replace = "")
    d$pos = gsub(x = chr_pos, pattern = ".*:", replace = "")
    d = d[which(d$chr %in% 1:22),]
    return(d)
}

to_snp = function(d){
    d = d %>% filter(nchar(alleles) == 9)
    d$ref = substr(d$alleles, 3,3)
    d$alt = substr(d$alleles, 7,7)
    return(d)
}

format_difs_header = function(d, p1, p2, o, f){
    i1 = ifelse(names(d)[3] == p1, "par1_gt", ifelse(names(d)[3] == p2, "par2_gt", "off_gt"))
    i2 = ifelse(names(d)[4] == p1, "par1_gt", ifelse(names(d)[4] == p2, "par2_gt", "off_gt"))
    i3 = ifelse(names(d)[5] == p1, "par1_gt", ifelse(names(d)[5] == p2, "par2_gt", "off_gt"))
    names(d) = c("locus", "alleles", i1, i2, i3)
    d$par1 = p1
    d$par2 = p2
    d$off = o
    d$fid = f
    d = format_difs(d)
    return(d)
}

identify_dnm = function(difs, difs_alt_alleles){    
    # next, go across 0/0 0/0 0/1. 
    # first, remove anything that appears twice, these are sites where the offspring differs from parents at more than one site
    duplicated_indices = which(duplicated(difs$locus) | duplicated(difs$locus, fromLast = TRUE))
    difs = difs[-duplicated_indices,]
    # next, filter to snps 
    difs_snp = to_snp(difs)
    difs_snp$from = difs_snp$ref
    difs_snp$to = difs_snp$alt
    difs_snp$remove=FALSE
    # next, go across 0/0 0/0 0/1
    for (i in 1:nrow(difs_snp)){
        # if the locus appears in difs_alt_alleles more than once, remove it
        sub = which(difs_alt_alleles$locus == difs_snp$locus[i])
        if (length(sub) > 1){
            difs_snp$remove[i] = TRUE
        }
    }
    
    difs_snp = difs_snp %>% filter(!remove)
    return(difs_snp)
}

In [ ]:
for (i in 1:nrow(trios)){
    if (i %% 100 == 0){
        message(i)
    }
    par1 = trios$par1[i]
    par2 = trios$par2[i]
    off = trios$offspring[i]
    fid = trios$fid[i]
    f = paste("./trios/difs/", par1, "_", par2, "_", off, "_difs.tsv", sep = "")
    f_alt_alleles = paste("./trios/difs_all_alt/", par1, "_", par2, "_", off, "_alt_all_alt.tsv", sep = "")
    f_out_snps = paste("./trios/formatted_difs/", par1, "_", par2, "_", off, "_snps_formatted.tsv", sep = "")
    if (file.exists(f) & !file.exists(f_out_snps)){
        trios$difs_file_exists[i] = TRUE
        d = fread(f, sep = "\t", header = TRUE)
        d = format_difs_header(d, par1, par2, off, fid)
        d_alt_alleles = fread(f_alt_alleles, sep = "\t", header = TRUE)
        d_alt_alleles = format_difs_header(d_alt_alleles, par1, par2, off, fid)
        d_all = identify_dnm(d, d_alt_alleles)
        trios$num_snps_raw[i] = nrow(d_all)
        fwrite(d_all, f_out_snps, sep = "\t", quote = FALSE, row.names = FALSE) 
    }
}

In [ ]:
i = 1
par1 = trios$par1[i]
par2 = trios$par2[i]
off = trios$off[i]
trios$QC_snps = NA
f_out_snps = paste("./trios/formatted_difs/", par1, "_", par2, "_", off, "_snps_formatted.tsv", sep = "")
all_snps = fread(f_out_snps, header = TRUE) %>% select(locus, alleles, par1_gt, par2_gt, off_gt, par1, 
                                                       par2, off, ref, alt)
trios$QC_snps[1] = nrow(all_snps)
for (i in 2:nrow(trios)){
    par1 = trios$par1[i]
    par2 = trios$par2[i]
    off = trios$off[i]
    f_out_snps = paste("./trios/formatted_difs/", par1, "_", par2, "_", off, "_snps_formatted.tsv", sep = "")
    sub = fread(f_out_snps, header = TRUE) %>% select(locus, alleles, par1_gt, par2_gt, off_gt, par1, 
                                                       par2, off, ref, alt)
    trios$QC_snps[i] = nrow(sub)
    all_snps = rbind(all_snps, sub)
}

chr_pos = gsub(x = all_snps$locus, pattern = "chr", replace = "")
all_snps$chr = gsub(x = chr_pos, pattern = ":.*", replace = "")
all_snps$pos = gsub(x = chr_pos, pattern = ".*:", replace = "")

fwrite(all_snps, "./trios_snps_after_standard_QC_feb7.txt", sep = "\t", quote = FALSE, row.names = FALSE)

In [ ]:
filters = fread("./giab_difficult_merged_oct27.bed")
all_snps_bed = data.frame(chrom = all_snps$chr, start = all_snps$pos, end = as.numeric(all_snps$pos)+1)
all_snps_bed = cbind(all_snps_bed, all_snps)
snps_filter = bedtoolsr::bt.intersect(a = all_snps_bed, b = filters, c = TRUE)
names(snps_filter) = c("chrom", "start", "end", names(all_snps), "in_giab")
snps_filtered = snps_filter %>% filter(in_giab == 0)
fwrite(snps_filtered, "./trios_snps_after_standard_QC_GIAB.txt", sep = "\t", quote = FALSE, row.names = FALSE) 

In [ ]:
# now quads, so redefine functions

In [ ]:
quad = fread("./relatedness_dec20/quads_new.txt", sep = "\t")
flagged = fread("./flagged_samples.tsv", sep = "\t", header= TRUE)
quad = quad %>% filter(!(off1 %in% flagged$s) & !(off2 %in% flagged$s) & !(par1 %in% flagged$s) & !(par2 %in% flagged$s) )

In [ ]:
format_difs = function(difs){
    names(difs) = c("locus", "alleles", "idv1_gt", "idv2_gt")
    duplicated_indices = which(duplicated(difs$locus) | duplicated(difs$locus, fromLast = TRUE))
    difs = difs[-duplicated_indices,]
    difs = difs %>% filter(nchar(alleles) == 9)
    difs = difs[which(mapply(is_00_01, difs$idv1_gt, difs$idv2_gt)),]
    difs$chr = NA
    difs$pos = NA
    chr_pos = gsub(x = difs$locus, pattern = "chr", replace = "")
    difs$chr = gsub(x = chr_pos, pattern = ":.*", replace = "")
    difs$pos = gsub(x = chr_pos, pattern = ".*:", replace = "") 
    difs = difs %>% filter(chr %in% 1:22)
    return(difs)
}

In [ ]:
is_00_01 = function(i1, i2){
    return((i1 %in% c("0/0", "0|0") & i2 %in% c("0/1", "0|1", "1/0", "1|0")) |
    (i2 %in% c("0/0", "0|0") & i1 %in% c("0/1", "0|1", "1/0", "1|0")))
}

In [ ]:
quad$file_exists = NA
for (i in 1:nrow(quad)){
    message(i)
    idv1 = quad$off1[i]
    idv2 = quad$off2[i]
    f_all_alt = paste("./sibs/difs_all_alt/", idv1, "_", idv2, "_difs_all_alt.tsv", sep = "")
    f_out = paste("./sibs/quads_formatted_difs/", idv1, "_", idv2, "_difs_all_alt_ibd2_formatted.tsv", sep = "")
    if (file.exists(f_all_alt)){
        quad$file_exists[i] = TRUE
        d_all_alt = fread(f_all_alt, sep = "\t", header = TRUE)
        formatted = format_difs(d_all_alt)
        i1 = idv1
        i2 = idv2
        fwrite(formatted, f_out, sep = "\t", quote = FALSE, row.names = FALSE)  
    } else {
        quad$file_exists[i] = FALSE
    }
}

In [ ]:
quad$file_exists = NA
quad$num_snps_QC = NA
for (i in 1:1){
    idv1 = quad$off1[i]
    idv2 = quad$off2[i]
    f = paste("./sibs/quads_formatted_difs/", idv1, "_", idv2, "_difs_all_alt_ibd2_formatted.tsv", sep = "")
    if (file.exists(f)){
        snps = fread(f, sep = "\t", header = TRUE)
        quad$num_snps_QC[i] = nrow(snps)
        quad$file_exists[i] = TRUE
    } else {
        quad$file_exists[i] = FALSE
    }
}

all_snps = snps 
all_snps$idv1 = idv1
all_snps$idv2 = idv2
all_snps$fid = quad$fid[1]

for (i in 2:nrow(quad)){
    if (i %% 20 == 0){
        message(i)
    }
    idv1 = quad$off1[i]
    idv2 = quad$off2[i]
    f = paste("./sibs/quads_formatted_difs/", idv1, "_", idv2, "_difs_all_alt_ibd2_formatted.tsv", sep = "")
    if (file.exists(f)){
        snps = fread(f, sep = "\t", header = TRUE)
        quad$num_snps_QC[i] = nrow(snps)
        quad$file_exists[i] = TRUE
        snps$idv1 = idv1 
        snps$idv2 = idv2
        snps$fid = quad$fid[i]
        all_snps = rbind(all_snps, snps)
    } else {
        quad$file_exists[i] = FALSE
    }
}

fwrite(all_snps, "./quad_sibs_all_snps_after_standard_QC.txt", sep = "\t",
      row.names = FALSE, col.names = TRUE, quote = FALSE)
all_snps_bed = data.frame("chrom" = all_snps$chr, "start" = all_snps$pos, "end" = all_snps$pos+1)
all_snps_giab = bedtoolsr::bt.intersect(a = all_snps_bed, b = giab, c = TRUE)
all_snps$in_giab = all_snps_giab$V4
all_snps_giab_filtered = all_snps %>% filter(in_giab == 0)
fwrite(all_snps_giab_filtered, "./quad_sibs_all_snps_after_standard_QC_GIAB.txt", sep = "\t",
      row.names = FALSE, col.names = TRUE, quote = FALSE)